# Phase 3 — Modélisation : Benchmark de classification

**Objectif :** Entraîner et comparer tous les modèles sur la tâche de prédiction de direction J+1.

**Input :** `data/X_train.pkl`, `data/X_val.pkl`, `data/X_test.pkl`, `data/y_*.pkl` (générés par `02_preprocessing.py`)

**Métrique principale :** F1-score (classe 1 = hausse)

In [1]:
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, "modèle llm")

DATA_DIR   = Path("data")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# Chargement des splits
def load_pkl(name):
    with open(DATA_DIR / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

X_train = load_pkl("X_train")
X_val   = load_pkl("X_val")
X_test  = load_pkl("X_test")
y_train = load_pkl("y_train")
y_val   = load_pkl("y_val")
y_test  = load_pkl("y_test")

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}  |  hausse : {y_train.mean():.2%}")
print(f"X_val   : {X_val.shape}  |  y_val   : {y_val.shape}  |  hausse : {y_val.mean():.2%}")
print(f"X_test  : {X_test.shape}  |  y_test  : {y_test.shape}  |  hausse : {y_test.mean():.2%}")
print(f"\nFeatures ({X_train.shape[1]}) :")
print(list(X_train.columns))

X_train : (1177, 74)  |  y_train : (1177,)  |  hausse : 50.64%
X_val   : (365, 74)  |  y_val   : (365,)  |  hausse : 53.97%
X_test  : (821, 74)  |  y_test  : (821,)  |  hausse : 50.43%

Features (74) :
['ret_1d_btc', 'ret_3d_btc', 'ret_7d_btc', 'ret_14d_btc', 'ret_30d_btc', 'vol_7d_btc', 'vol_30d_btc', 'ret_1d_gold', 'ret_3d_gold', 'ret_7d_gold', 'ret_14d_gold', 'ret_30d_gold', 'vol_7d_gold', 'vol_30d_gold', 'ret_1d_eth', 'ret_3d_eth', 'ret_7d_eth', 'ret_14d_eth', 'ret_30d_eth', 'vol_7d_eth', 'vol_30d_eth', 'ret_1d_sp500', 'ret_3d_sp500', 'ret_7d_sp500', 'ret_14d_sp500', 'ret_30d_sp500', 'vol_7d_sp500', 'vol_30d_sp500', 'ret_1d_dxy', 'ret_3d_dxy', 'ret_7d_dxy', 'ret_14d_dxy', 'ret_30d_dxy', 'vol_7d_dxy', 'vol_30d_dxy', 'ret_1d_vix', 'ret_3d_vix', 'ret_7d_vix', 'ret_14d_vix', 'ret_30d_vix', 'vol_7d_vix', 'vol_30d_vix', 'ret_1d_us10y', 'ret_3d_us10y', 'ret_7d_us10y', 'ret_14d_us10y', 'ret_30d_us10y', 'vol_7d_us10y', 'vol_30d_us10y', 'ret_1d_oil', 'ret_3d_oil', 'ret_7d_oil', 'ret_14d_oil'

## Groupe 1 — Baselines sklearn

Les modèles sklearn n'utilisent pas de séquences temporelles : ils voient chaque jour comme un vecteur de features indépendant. Validation par `TimeSeriesSplit` pour respecter la chronologie.

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

# Concaténation train+val pour la cross-validation temporelle
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

tscv = TimeSeriesSplit(n_splits=5)

sklearn_models = {
    "Dummy (most_frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression":   LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":         RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting":     GradientBoostingClassifier(n_estimators=300, random_state=42),
}

sklearn_results = {}
print(f"{'Modèle':<25} {'CV F1 (mean)':>12} {'CV F1 (std)':>11}")
print("-" * 50)
for name, clf in sklearn_models.items():
    cv_f1 = cross_val_score(clf, X_trainval, y_trainval,
                            cv=tscv, scoring="f1", n_jobs=-1)
    sklearn_results[name] = {"cv_f1_mean": cv_f1.mean(), "cv_f1_std": cv_f1.std()}
    print(f"{name:<25} {cv_f1.mean():>12.4f} ± {cv_f1.std():.4f}")

Modèle                    CV F1 (mean) CV F1 (std)
--------------------------------------------------
Dummy (most_frequent)           0.5372 ± 0.2692
Logistic Regression             0.4555 ± 0.1791
Random Forest                   0.4682 ± 0.1339
Gradient Boosting               0.4873 ± 0.0522


In [3]:
# Entraînement final sur train+val et évaluation sur val (pour sélection)
sklearn_val_results = {}
for name, clf in sklearn_models.items():
    clf.fit(X_train, y_train)
    y_prob = clf.predict_proba(X_val)[:, 1] if hasattr(clf, "predict_proba") else clf.predict(X_val)
    y_pred = (y_prob >= 0.5).astype(int)
    sklearn_val_results[name] = {
        "F1":       f1_score(y_val, y_pred, zero_division=0),
        "AUC-ROC":  roc_auc_score(y_val, y_prob),
        "Accuracy": accuracy_score(y_val, y_pred),
    }

pd.DataFrame(sklearn_val_results).T.sort_values("F1", ascending=False)

,F1,AUC-ROC,Accuracy
Dummy (most_frequent),0.701068,0.500000,0.539726
Random Forest,0.497110,0.535609,0.523288
Gradient Boosting,0.467692,0.555958,0.526027
Logistic Regression,0.000000,0.502478,0.460274


## Groupe 2 — Deep Learning PyTorch (séquences temporelles)

Les modèles DL utilisent des séquences de 60 jours. Input shape : `(n_samples, 60, n_features)`.

In [4]:
from scripts_models.config import SEQUENCE_LENGTH, EPOCHS, LEARNING_RATE, BATCH_SIZE, EARLY_STOPPING_PATIENCE
from scripts_models.config import LSTM_CONFIG, GRU_CONFIG, CNN_LSTM_CONFIG, TRANSFORMER_CONFIG, TFT_CONFIG, XGBOOST_CONFIG
from scripts_models.trainer import Trainer
from scripts_models.metrics import evaluate_model, print_evaluation

SEQ_LEN = SEQUENCE_LENGTH  # 60 jours

def make_sequences(X: pd.DataFrame, y: pd.Series, seq_len: int):
    X_arr = X.values
    y_arr = y.values
    Xs, ys = [], []
    for i in range(seq_len, len(X_arr)):
        Xs.append(X_arr[i - seq_len:i])
        ys.append(y_arr[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_tr_seq, y_tr_seq = make_sequences(X_train, y_train, SEQ_LEN)
X_vl_seq, y_vl_seq = make_sequences(X_val,   y_val,   SEQ_LEN)
X_te_seq, y_te_seq = make_sequences(X_test,  y_test,  SEQ_LEN)

n_features = X_tr_seq.shape[2]
print(f"Séquences — Train: {X_tr_seq.shape} | Val: {X_vl_seq.shape} | Test: {X_te_seq.shape}")
print(f"n_features = {n_features}")

Séquences — Train: (1117, 60, 74) | Val: (305, 60, 74) | Test: (761, 60, 74)
n_features = 74


In [5]:
from scripts_models.lstm_model import LSTMModel
from scripts_models.gru_model import GRUModel
from scripts_models.cnn_lstm_model import CNNLSTMModel
from scripts_models.transformer_model import TransformerModel
from scripts_models.tft_model import SimplifiedTFT
from scripts_models.xgboost_model import XGBoostWrapper
import time

dl_results = {}

def run_dl_model(model, name):
    print(f"\n{'='*55}\n  {name}\n{'='*55}")
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Paramètres : {n_params:,}")

    trainer = Trainer(model=model, learning_rate=LEARNING_RATE,
                      batch_size=BATCH_SIZE, epochs=EPOCHS,
                      patience=EARLY_STOPPING_PATIENCE)
    t0 = time.time()
    trainer.train(X_tr_seq, y_tr_seq, X_vl_seq, y_vl_seq, verbose=True)
    elapsed = time.time() - t0

    y_prob_val  = trainer.predict(X_vl_seq)
    results_val = evaluate_model(y_vl_seq, y_prob_val, name)
    results_val["n_params"] = n_params
    results_val["time_s"]   = round(elapsed, 1)
    print_evaluation(results_val)
    dl_results[name] = {"trainer": trainer, "metrics_val": results_val}
    return trainer

# --- LSTM ---
run_dl_model(LSTMModel(n_features, LSTM_CONFIG["hidden_size"],
                        LSTM_CONFIG["num_layers"], LSTM_CONFIG["dropout"]), "LSTM")

# --- BiLSTM ---
run_dl_model(LSTMModel(n_features, LSTM_CONFIG["hidden_size"],
                        LSTM_CONFIG["num_layers"], LSTM_CONFIG["dropout"],
                        bidirectional=True), "BiLSTM")

# --- GRU ---
run_dl_model(GRUModel(n_features, GRU_CONFIG["hidden_size"],
                       GRU_CONFIG["num_layers"], GRU_CONFIG["dropout"]), "GRU")

# --- CNN-LSTM ---
run_dl_model(CNNLSTMModel(n_features, CNN_LSTM_CONFIG["cnn_filters"],
                           CNN_LSTM_CONFIG["cnn_kernel_size"],
                           CNN_LSTM_CONFIG["lstm_hidden_size"],
                           CNN_LSTM_CONFIG["lstm_num_layers"],
                           CNN_LSTM_CONFIG["dropout"]), "CNN-LSTM")

# --- Transformer ---
run_dl_model(TransformerModel(n_features, TRANSFORMER_CONFIG["d_model"],
                               TRANSFORMER_CONFIG["nhead"],
                               TRANSFORMER_CONFIG["num_encoder_layers"],
                               TRANSFORMER_CONFIG["dim_feedforward"],
                               TRANSFORMER_CONFIG["dropout"]), "Transformer")

# --- TFT ---
run_dl_model(SimplifiedTFT(n_features, TFT_CONFIG["hidden_size"],
                            TFT_CONFIG["lstm_layers"],
                            TFT_CONFIG["attention_heads"],
                            TFT_CONFIG["dropout"]), "TFT")


  LSTM
  Paramètres : 244,865
  Epoch  10/100 | Train Loss: 0.595020 | Val Loss: 0.735298 | LR: 5.0e-04 | Time: 14.1s
  → Early stopping à l'époque 16
  → Entraînement terminé en 22.4s
  → Meilleure val loss: 0.693132

  Résultats : LSTM
  F1-score (classe 1)  : 0.5125  ← métrique principale
  Accuracy             : 0.4885
  AUC-ROC              : 0.4883
  Precision            : 0.5190
  Recall               : 0.5062
  Confusion Matrix     :
    TN=   67  FP=   76
    FN=   80  TP=   82


  BiLSTM
  Paramètres : 620,673
  Epoch  10/100 | Train Loss: 0.521947 | Val Loss: 0.805861 | LR: 5.0e-04 | Time: 27.6s
  → Early stopping à l'époque 16
  → Entraînement terminé en 43.1s
  → Meilleure val loss: 0.693353

  Résultats : BiLSTM
  F1-score (classe 1)  : 0.5501  ← métrique principale
  Accuracy             : 0.4852
  AUC-ROC              : 0.4846
  Precision            : 0.5134
  Recall               : 0.5926
  Confusion Matrix     :
    TN=   52  FP=   91
    FN=   66  TP=   96


  GRU
 

In [ ]:
# --- XGBoost ---
print(f"\n{'='*55}\n  XGBoost\n{'='*55}")
xgb_wrapper = XGBoostWrapper(XGBOOST_CONFIG)
xgb_wrapper.fit(X_tr_seq, y_tr_seq, X_vl_seq, y_vl_seq)
y_prob_xgb = xgb_wrapper.predict(X_vl_seq)
xgb_metrics = evaluate_model(y_vl_seq, y_prob_xgb, "XGBoost")
print_evaluation(xgb_metrics)
dl_results["XGBoost"] = {"trainer": xgb_wrapper, "metrics_val": xgb_metrics}

## Tableau comparatif (validation set)

In [ ]:
all_val = {}
for name, res in sklearn_val_results.items():
    all_val[name] = res
for name, res in dl_results.items():
    m = res["metrics_val"]
    all_val[name] = {"F1": m["F1"], "AUC-ROC": m["AUC-ROC"], "Accuracy": m["Accuracy"]}

results_df = pd.DataFrame(all_val).T.sort_values("F1", ascending=False)
print("=== Performances sur le VAL SET (2023) ===")
print(results_df.round(4).to_string())

best_name = results_df[~results_df.index.str.contains("Dummy")].index[0]
print(f"\n→ Meilleur modèle sur val : {best_name}  (F1 = {results_df.loc[best_name, 'F1']:.4f})")

## Évaluation finale sur le TEST SET

⚠️ Cette cellule ne doit être exécutée **qu'une seule fois**, après avoir sélectionné le meilleur modèle sur le val set. Toute optimisation post-test introduit du leakage.

In [ ]:
import torch

# Évaluation du meilleur modèle sur le test set
best_obj = dl_results[best_name]["trainer"]

if isinstance(best_obj, XGBoostWrapper):
    y_prob_test = best_obj.predict(X_te_seq)
else:
    y_prob_test = best_obj.predict(X_te_seq)

test_results = evaluate_model(y_te_seq, y_prob_test, f"{best_name} [TEST]")
print_evaluation(test_results)

# Sauvegarde du meilleur modèle
best_path = MODELS_DIR / "best_model.pkl"
with open(best_path, "wb") as f:
    if isinstance(best_obj, XGBoostWrapper):
        pickle.dump(best_obj, f)
    else:
        torch.save(best_obj.model.state_dict(), str(best_path).replace(".pkl", ".pt"))
        pickle.dump({"name": best_name, "n_features": n_features}, f)

print(f"\nMeilleur modèle sauvegardé → {best_path}")